[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_20/2_sql.ipynb)

# Day 20: SQL (medium)

**How to work:** Read the tables first (columns, one row = what?). Write the query in your head in the order FROM, WHERE, GROUP BY, HAVING, SELECT, ORDER BY. Check the result for duplicates and NULLs before you open the solution.

**Today (30 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | Retries in the event pipeline | medium | 5 |
| Q2 | Double taps | medium | 7 |
| Q3 | One row per user in the experiment | medium | 6 |
| Q4 | The same app listed many times | medium | 7 |
| Q5 | Top apps per category after cleaning (review) | medium | 5 |

**Handbook:** [SQL](https://mahsa-agz.github.io/ds-handbook/#sql), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Setup

Run this cell once before the questions.

In [ ]:
import sqlite3
import pandas as pd

def run(q, con=None):
    """Runs a query and shows the result as a table (SELECT / WITH); other statements just run."""
    con = con or db
    if not q.strip():
        print("Write your query between the triple quotes, then run the cell.")
        return None
    if q.strip().lower().startswith(("select", "with")):
        return pd.read_sql_query(q, con)
    con.executescript(q)
    print("done")

# Finds the data folder (local copy of the repository, or /content/data in Colab) and downloads a missing file
# from the ds-handbook repository on GitHub (or from its original source).
import os, sys, urllib.request

RAW = "https://raw.githubusercontent.com/mahsa-agz/mahsa-agz.github.io/main/ds-handbook/exam_prep/data/"

def _find_data():
    if "google.colab" in sys.modules:
        os.makedirs("/content/data", exist_ok=True)
        return "/content/data"
    for p in ("../data", "data", "exam_prep/data"):
        if os.path.isdir(p):
            return p
    os.makedirs("data", exist_ok=True)
    return "data"

DATA = _find_data()
print("data folder:", DATA)

def data_path(name, url=None):
    """Local path of a data file; downloads it from the repository (or from url) if it is not there yet."""
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        for source in (RAW + name, url):
            if not source:
                continue
            try:
                print("downloading", name)
                urllib.request.urlretrieve(source, path)
                break
            except Exception as e:
                print("  could not download from", source, "->", e)
    return path

db = sqlite3.connect(':memory:')

# Made-up shopping-app tables (generated with a fixed seed, so everyone gets the same rows).
import numpy as np

def make_app_tables(con, seed=7):
    rs = np.random.RandomState(seed)
    start, end = pd.Timestamp("2024-01-01"), pd.Timestamp("2024-03-31")
    n = 1500
    countries, c_p = ["US", "KR", "BR", "ID", "DE"], [0.30, 0.25, 0.20, 0.15, 0.10]
    platforms, p_p = ["ios", "android", "web"], [0.40, 0.45, 0.15]
    channels, ch_p = ["organic", "paid_social", "search", "referral"], [0.35, 0.30, 0.20, 0.15]
    signup_off = np.sort(rs.randint(0, 60, n))                     # day offsets 0..59 (Jan 1 to Feb 29)
    country = rs.choice(len(countries), n, p=c_p)
    platform = rs.choice(len(platforms), n, p=p_p)
    channel = rs.choice(len(channels), n, p=ch_p)
    referred_by = [None] * n
    for i in range(n):
        if channels[channel[i]] == "referral":
            earlier = np.nonzero(signup_off[:i] < signup_off[i])[0]
            if len(earlier):
                referred_by[i] = int(earlier[rs.randint(len(earlier))]) + 1
            else:
                channel[i] = 0                                       # nobody to refer them: organic
    users = pd.DataFrame({
        "user_id": np.arange(1, n + 1),
        "signup_date": [(start + pd.Timedelta(days=int(d))).strftime("%Y-%m-%d") for d in signup_off],
        "country": [countries[c] for c in country],
        "platform": [platforms[p] for p in platform],
        "channel": [channels[c] for c in channel],
        "referred_by": pd.array(referred_by, dtype="Int64"),
    })
    plat_boost = {"ios": 0.08, "android": 0.0, "web": -0.10}
    chan_boost = {"organic": 0.05, "paid_social": -0.08, "search": 0.0, "referral": 0.08}
    act, ev, buys = [], [], []
    total_days = (end - start).days + 1
    for i in range(n):
        uid = i + 1
        p = rs.beta(2, 3) + plat_boost[platforms[platform[i]]] + chan_boost[channels[channel[i]]]
        p = min(max(p, 0.03), 0.9)
        life = 1 if rs.rand() < 0.30 else 1 + int(rs.exponential(35))   # days until the user stops coming back
        buyer = rs.rand() < 0.55
        prev = True
        for t in range(0, total_days - signup_off[i]):
            if t == 0:
                active = True
            elif t >= life:
                active = rs.rand() < 0.01                                 # rare comeback after churn
            else:
                active = rs.rand() < (min(p + 0.25, 0.95) if prev else p * 0.7)
            prev = active
            if not active:
                continue
            day = start + pd.Timedelta(days=int(signup_off[i] + t))
            sessions = 1 + rs.poisson(0.8)
            minutes = int(sum(rs.randint(2, 25) for _ in range(sessions)))
            act.append((uid, day.strftime("%Y-%m-%d"), sessions, minutes))
            clock = day + pd.Timedelta(seconds=int(rs.randint(6 * 3600, 22 * 3600)))
            if rs.rand() < 0.65:
                steps = ["view_item"]
                if rs.rand() < 0.40:
                    steps.append("add_to_cart")
                    if rs.rand() < 0.55:
                        steps.append("checkout")
                        if buyer and rs.rand() < 0.75:
                            steps.append("purchase")
                elif rs.rand() < 0.03:
                    steps = ["add_to_cart"]                               # from the wishlist, no view that day
                for s in steps:
                    clock = clock + pd.Timedelta(seconds=int(rs.randint(15, 900)))
                    ev.append((uid, clock.strftime("%Y-%m-%d %H:%M:%S"), s))
                    if s == "purchase":
                        amount = round(float(rs.gamma(2.0, 18.0)) + 5, 2)
                        status = "refunded" if rs.rand() < 0.07 else "completed"
                        buys.append((uid, clock.strftime("%Y-%m-%d %H:%M:%S"), amount, status))
    activity = pd.DataFrame(act, columns=["user_id", "activity_date", "sessions", "minutes"])
    events = pd.DataFrame(ev, columns=["user_id", "event_time", "event_name"])
    events = events.sort_values(["event_time", "user_id"], kind="mergesort").reset_index(drop=True)
    events.insert(0, "event_id", np.arange(1, len(events) + 1))
    purchases = pd.DataFrame(buys, columns=["user_id", "order_time", "amount", "status"])
    purchases = purchases.sort_values(["order_time", "user_id"], kind="mergesort").reset_index(drop=True)
    purchases.insert(0, "order_id", np.arange(1, len(purchases) + 1))
    # raw event log as it arrives from the apps: retries create exact copies, double taps create near copies
    raw = events.copy()
    dup = raw[rs.rand(len(raw)) < 0.04]
    tap = raw[rs.rand(len(raw)) < 0.01].copy()
    tap["event_time"] = (pd.to_datetime(tap["event_time"]) +
                         pd.to_timedelta(rs.randint(1, 4, len(tap)), unit="s")).dt.strftime("%Y-%m-%d %H:%M:%S")
    raw = pd.concat([raw, dup, tap]).sort_values(["event_time", "user_id"], kind="mergesort").reset_index(drop=True)
    raw = raw.drop(columns="event_id")
    raw.insert(0, "raw_id", np.arange(1, len(raw) + 1))
    raw["received_at"] = (pd.to_datetime(raw["event_time"]) +
                          pd.to_timedelta(rs.randint(1, 600, len(raw)), unit="s")).dt.strftime("%Y-%m-%d %H:%M:%S")
    for name, df in [("app_users", users), ("daily_activity", activity), ("app_events", events),
                     ("purchases", purchases), ("app_events_raw", raw)]:
        df.to_sql(name, con, index=False, if_exists="replace")

make_app_tables(db)

# Real data: Udacity landing-page A/B test log (educational use; has deliberate data problems).
pd.read_csv(data_path("ab_data.csv", "https://raw.githubusercontent.com/F-Zarian/Analyze_AB_test_results/main/ab_data.csv")).rename(columns={"group": "grp"}).to_sql("ab_data", db, index=False)

# Real data: Google Play store apps (scraped; study use only). Messy text columns on purpose.
_g = pd.read_csv(data_path("googleplaystore.csv", "https://raw.githubusercontent.com/malborroni/Foundations_of_Computer-Science/master/datasets/googleplaystore.csv")).rename(columns=lambda c: c.lower().replace(" ", "_"))
_g["app"] = _g["app"].str.replace("\u2013", "-").str.replace("\u2014", "-")   # long dashes in app names -> "-"
_g.to_sql("play_apps", db, index=False)

print("tables:", ", ".join(r[0] for r in db.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name")))

## Tables

**Made-up shopping-app tables** (invented for practice, generated with a fixed seed: 1,500 users signed up
2024-01-01 to 2024-02-29, activity until 2024-03-31):

| Table | One row is | Columns |
|---|---|---|
| `app_users` | one user | `user_id`, `signup_date` ('YYYY-MM-DD'), `country` (US/KR/BR/ID/DE), `platform` (ios/android/web), `channel` (organic/paid_social/search/referral), `referred_by` (user_id of the referrer, NULL unless channel = referral) |
| `daily_activity` | one user on one day they opened the app | `user_id`, `activity_date` ('YYYY-MM-DD'), `sessions`, `minutes`. No row = not active that day |
| `app_events` | one shopping event | `event_id`, `user_id`, `event_time` ('YYYY-MM-DD HH:MM:SS'), `event_name` (view_item / add_to_cart / checkout / purchase) |
| `purchases` | one order | `order_id`, `user_id`, `order_time`, `amount` (USD), `status` (completed / refunded) |
| `app_events_raw` | one event as received from the apps (has retries and double taps) | `raw_id`, `user_id`, `event_time`, `event_name`, `received_at` |

**Real table: ab_data** (Udacity landing-page experiment, January 2017; contains deliberate mistakes):

| Table | One row is | Columns |
|---|---|---|
| `ab_data` | one page visit (294,478) | `user_id`, `timestamp` (text with microseconds), `grp` ('control' / 'treatment'; renamed from `group`, a reserved word), `landing_page` ('old_page' / 'new_page'), `converted` (0/1) |

**Real table: Google Play apps** (scraped in 2018; messy on purpose):

| Table | One row is | Columns |
|---|---|---|
| `play_apps` | one scraped app listing (10,841) | `app`, `category`, `rating` (can be NULL), `reviews` (text!), `size`, `installs` (text like '10,000+'), `type`, `price`, `content_rating`, `genres`, `last_updated` (text like 'January 7, 2018'), `current_ver`, `android_ver` |

The queries run in SQLite. They are written in standard SQL that also works in PostgreSQL; differences (dates, integer division) are noted in the solutions.

---
### Q1. Retries in the event pipeline

*medium, about 5 min*

Shopping app. `app_events_raw` is the event log as received from the phones. When the network
fails, the app **retries** and the same event arrives twice: same `user_id`, `event_time` and `event_name`, but a new
`raw_id` and a later `received_at`.

Return one summary row: `raw_rows`, `unique_events` (keep one row per user, time and event name: the one received
first), `copies_removed`, `events_with_copies` and `max_copies` (most rows for one event).

Example: rows (7, 10:00:00, view_item, received 10:00:05) and (7, 10:00:00, view_item, received 10:03:00) are one
event; keep the first.

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: "same business key, several rows, keep one". Pattern: ROW_NUMBER over `PARTITION BY` the business key, ordered by the tie rule; keep rn = 1.

</details>

<details><summary><b>Hint 2</b></summary>

1. `ROW_NUMBER() OVER (PARTITION BY user_id, event_time, event_name ORDER BY received_at, raw_id) AS rn`.
2. `COUNT(*) OVER (same partition) AS copies`.
3. Summarise with conditional sums: rn = 1, rn > 1, rn = 1 AND copies > 1.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH ranked AS (
    SELECT raw_id, user_id, event_time, event_name, received_at,
           ROW_NUMBER() OVER (PARTITION BY user_id, event_time, event_name
                              ORDER BY received_at, raw_id) AS rn,
           COUNT(*)     OVER (PARTITION BY user_id, event_time, event_name) AS copies
    FROM app_events_raw
)
SELECT COUNT(*)                                           AS raw_rows,
       SUM(CASE WHEN rn = 1 THEN 1 ELSE 0 END)            AS unique_events,
       SUM(CASE WHEN rn > 1 THEN 1 ELSE 0 END)            AS copies_removed,
       SUM(CASE WHEN rn = 1 AND copies > 1 THEN 1 ELSE 0 END) AS events_with_copies,
       MAX(copies)                                        AS max_copies
FROM ranked
```

Result:

| raw_rows | unique_events | copies_removed | events_with_copies | max_copies |
|---|---|---|---|---|
| 20082 | 19287 | 795 | 795 | 2 |

**Why:** ROW_NUMBER keeps exactly one row per key and lets you choose which (first received, then lowest raw_id as a final tie breaker); `SELECT DISTINCT` would also work here, but only when every column is identical, and it cannot choose between rows that differ in a non-key column such as received_at. 795 of 20,082 rows are retries. Always say the business key out loud: here it is (user, time, event), not raw_id, which is unique by construction. PostgreSQL also has `DISTINCT ON (key) ... ORDER BY key, received_at`.

**Complexity:** One window sort over 20,082 rows.

**Common mistakes:** Partitioning by raw_id (nothing is a duplicate). `GROUP BY` the key with `MIN(received_at)` and then losing the other columns. No tie breaker, so the kept row changes between runs.

**Learn more:** [sql-dedup-cleaning](https://mahsa-agz.github.io/ds-handbook/#sql-dedup-cleaning), [sql-window-ranking](https://mahsa-agz.github.io/ds-handbook/#sql-window-ranking)

</details>

---
### Q2. Double taps

*medium, about 7 min*

After removing the retries from Q1, some events are still doubled: a user taps twice and the app logs
the **same `event_name`** for the **same user** again **1 to 3 seconds** later. Drop every event that comes at most
**3 seconds** after the previous kept-or-not event of the same user and event name.

Return `after_exact` (rows after Q1), `after_double_taps` and `reference_clean_table` (row count of the clean table
`app_events`, which should match if your cleaning is right).

Example: view_item at 10:00:00 and 10:00:02 by user 7: keep 10:00:00 only. A view_item at 10:00:30 stays.

Follow-up: why is `(julianday(t2) - julianday(t1)) * 86400 > 3` risky for this test?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: duplicates defined by "close in time", not by equal keys. Pattern: LAG per (user, event_name) ordered by time, drop rows whose gap is at most 3 seconds. It is the same gap logic as sessionization (day 19).

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE exact: Q1's ROW_NUMBER, keep rn = 1.
2. CTE lagged: `LAG(event_time) OVER (PARTITION BY user_id, event_name ORDER BY event_time, raw_id)`.
3. Keep rows where prev is NULL or the gap in whole seconds > 3: `strftime('%s', event_time) - strftime('%s', prev_time)`.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH exact AS (
    SELECT raw_id, user_id, event_time, event_name,
           ROW_NUMBER() OVER (PARTITION BY user_id, event_time, event_name
                              ORDER BY received_at, raw_id) AS rn
    FROM app_events_raw
), lagged AS (
    SELECT raw_id, user_id, event_time, event_name,
           LAG(event_time) OVER (PARTITION BY user_id, event_name ORDER BY event_time, raw_id) AS prev_time
    FROM exact
    WHERE rn = 1
), clean AS (
    SELECT *
    FROM lagged
    WHERE prev_time IS NULL
       OR strftime('%s', event_time) - strftime('%s', prev_time) > 3
)
SELECT (SELECT COUNT(*) FROM exact WHERE rn = 1) AS after_exact,
       COUNT(*)                                  AS after_double_taps,
       (SELECT COUNT(*) FROM app_events)         AS reference_clean_table
FROM clean
```

Result:

| after_exact | after_double_taps | reference_clean_table |
|---|---|---|
| 19287 | 19094 | 19094 |

**Why:** Cleaning in two passes: exact copies first (cheap and safe), then near copies with a rule you can explain (same user, same event, at most 3 s apart). The result matches the clean table exactly (19,094 rows), which is the check you would run in an exam if a reference exists, or by sampling rows by hand if not. Follow-up: julianday is a floating-point day number, so a gap of exactly 3 seconds can come out as 3.0000001 and pass `> 3`. On this data that version keeps 29 double taps. Compare integer seconds (`strftime('%s', ...)`), or in PostgreSQL `EXTRACT(EPOCH FROM t2 - t1)` on real timestamps. Note: comparing with the previous raw event (not the previous kept one) means a burst of taps 2 s apart for 10 s collapses to one event; say which rule you chose.

**Complexity:** Two window sorts over about 20k rows.

**Common mistakes:** Partitioning only by user (a view followed by a cart 2 s later is not a duplicate). Floating-point seconds. Doing the near-duplicate pass before the exact pass (works, but harder to reason about).

**Learn more:** [sql-dedup-cleaning](https://mahsa-agz.github.io/ds-handbook/#sql-dedup-cleaning), [sql-lag-lead](https://mahsa-agz.github.io/ds-handbook/#sql-lag-lead), [sql-gaps-islands](https://mahsa-agz.github.io/ds-handbook/#sql-gaps-islands)

</details>

---
### Q3. One row per user in the experiment

*medium, about 6 min*

`ab_data` (real) has 294,478 rows but fewer users: some users appear twice. Clean it in this order:
1. drop rows where group and page disagree (`(grp = 'treatment') <> (landing_page = 'new_page')`);
2. keep each user's **first** remaining row by `timestamp`.

Per `grp` return `users`, `conversions` and `conv_pct` (3 decimals).

Example: a user with rows (treatment, new_page, 09:00, converted 0) and (treatment, new_page, 11:00, converted 1)
keeps the 09:00 row: their exposure starts at the first visit.

Follow-up: why first visit and not "converted if any row converted"?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Pattern: filter, then ROW_NUMBER per user ordered by time, keep rn = 1, then aggregate.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE: `ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY timestamp)` over the consistent rows.
2. Outer: WHERE rn = 1, group by grp: COUNT(*), SUM(converted), AVG(converted).

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH clean AS (
    SELECT user_id, timestamp, grp, converted,
           ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY timestamp) AS rn
    FROM ab_data
    WHERE (grp = 'treatment') = (landing_page = 'new_page')
)
SELECT grp,
       COUNT(*) AS users,
       SUM(converted) AS conversions,
       ROUND(100.0 * AVG(converted), 3) AS conv_pct
FROM clean
WHERE rn = 1
GROUP BY grp
```

Result:

| grp | users | conversions | conv_pct |
|---|---|---|---|
| control | 145274 | 17489 | 12.039 |
| treatment | 145310 | 17264 | 11.881 |

**Why:** The order matters: dedup after removing inconsistent rows, otherwise a user's first row could be a bad one and the user would disappear. The text timestamp ('2017-01-21 22:11:48.556739') sorts correctly as text because it is zero padded. Result: 145,274 control and 145,310 treatment users, 12.039% versus 11.881%. Follow-up: the experiment unit is the user and the metric should be defined once per user; "any row converted" is also a valid per-user metric, but it gives users with two visits two chances, and two visits can be a consequence of the treatment. Decide the rule before looking at the results.

**Complexity:** One window sort over about 290k rows.

**Common mistakes:** Deduplicating before removing mismatches. `GROUP BY user_id` with `MAX(converted)` without thinking about the definition. Ordering by user_id instead of timestamp inside the partition.

**Learn more:** [sql-dedup-cleaning](https://mahsa-agz.github.io/ds-handbook/#sql-dedup-cleaning), [stats-ab-pitfalls](https://mahsa-agz.github.io/ds-handbook/#stats-ab-pitfalls)

</details>

---
### Q4. The same app listed many times

*medium, about 7 min*

`play_apps` (real scrape of Google Play) lists some apps several times (scraped at different moments,
sometimes under two categories), and `reviews` is **text**. Clean it:
1. keep only rows whose `reviews` is made of digits only (one shifted row has '3.0M');
2. keep **one row per app**: the one with the most reviews (as an integer); ties: the row that comes first in the
   table (`rowid`).

Return `raw_rows`, `bad_reviews_rows`, `distinct_apps`, `clean_rows`, and `dup_apps` (apps that had more than one
valid row).

Example: '8 Ball Pool' appears 7 times with reviews from 14,184,910 to 14,201,891; keep the 14,201,891 row.

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Pattern: validate the type first (GLOB for digits), then ROW_NUMBER per app ordered by the integer value DESC, keep rn = 1.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE typed: `WHERE reviews GLOB '[0-9]*' AND reviews NOT GLOB '*[^0-9]*'`, `CAST(reviews AS INTEGER)`.
2. CTE dedup: `ROW_NUMBER() OVER (PARTITION BY app ORDER BY reviews DESC, rid)` and `COUNT(*) OVER (PARTITION BY app)`.
3. One summary SELECT with scalar subqueries or conditional sums.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH typed AS (
    SELECT rowid AS rid, app, category, CAST(reviews AS INTEGER) AS reviews
    FROM play_apps
    WHERE reviews GLOB '[0-9]*' AND reviews NOT GLOB '*[^0-9]*'
), dedup AS (
    SELECT rid, app, category, reviews,
           ROW_NUMBER() OVER (PARTITION BY app ORDER BY reviews DESC, rid) AS rn,
           COUNT(*)     OVER (PARTITION BY app)                            AS n_rows
    FROM typed
)
SELECT (SELECT COUNT(*) FROM play_apps)                AS raw_rows,
       (SELECT COUNT(*) FROM play_apps) - (SELECT COUNT(*) FROM typed) AS bad_reviews_rows,
       (SELECT COUNT(DISTINCT app) FROM play_apps)     AS distinct_apps,
       SUM(CASE WHEN rn = 1 THEN 1 ELSE 0 END)         AS clean_rows,
       SUM(CASE WHEN rn = 1 AND n_rows > 1 THEN 1 ELSE 0 END) AS dup_apps
FROM dedup
```

Result:

| raw_rows | bad_reviews_rows | distinct_apps | clean_rows | dup_apps |
|---|---|---|---|---|
| 10841 | 1 | 9660 | 9659 | 798 |

**Why:** Type validation comes first: `CAST('3.0M' AS INTEGER)` silently returns 3 in SQLite (PostgreSQL would raise an error), so a bad row could win or lose a ranking for the wrong reason. The two GLOBs mean "starts with a digit and contains no non-digit". Then ROW_NUMBER per app keeps the most recent-looking snapshot (most reviews). More than a thousand rows are duplicates. Keeping one row per app on the app name alone also merges listings that sit in two categories; if category matters, decide which one wins and say it. PostgreSQL: `reviews ~ '^[0-9]+$'`.

**Complexity:** One window sort over about 10.8k rows.

**Common mistakes:** CAST without validation. `SELECT DISTINCT *` (rows differ in reviews, so nothing is removed). Keeping the first row instead of the most complete one.

**Learn more:** [sql-dedup-cleaning](https://mahsa-agz.github.io/ds-handbook/#sql-dedup-cleaning), [sql-window-ranking](https://mahsa-agz.github.io/ds-handbook/#sql-window-ranking)

</details>

---
### Q5. Top apps per category after cleaning (review)

*medium, about 5 min*

Using the cleaning rules of Q4 (digits-only reviews, one row per app with the most reviews), return
the **3 apps with the most reviews** in each of the categories `GAME`, `SOCIAL` and `COMMUNICATION` (ties share a
rank). Return `category`, `rank_in_cat`, `app`, `reviews`.

Why must the dedup happen before the ranking?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Pattern: dedup CTE (Q4), then DENSE_RANK per category, filter rank <= 3.

</details>

<details><summary><b>Hint 2</b></summary>

1. Reuse typed and dedup CTEs, keep rn = 1.
2. `DENSE_RANK() OVER (PARTITION BY category ORDER BY reviews DESC)`.
3. Filter the three categories and rank <= 3 in an outer query.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH typed AS (
    SELECT rowid AS rid, app, category, CAST(reviews AS INTEGER) AS reviews
    FROM play_apps
    WHERE reviews GLOB '[0-9]*' AND reviews NOT GLOB '*[^0-9]*'
), dedup AS (
    SELECT rid, app, category, reviews,
           ROW_NUMBER() OVER (PARTITION BY app ORDER BY reviews DESC, rid) AS rn
    FROM typed
), ranked AS (
    SELECT category, app, reviews,
           DENSE_RANK() OVER (PARTITION BY category ORDER BY reviews DESC) AS rank_in_cat
    FROM dedup
    WHERE rn = 1 AND category IN ('GAME', 'SOCIAL', 'COMMUNICATION')
)
SELECT category, rank_in_cat, app, reviews
FROM ranked
WHERE rank_in_cat <= 3
ORDER BY category, rank_in_cat, app
```

Result:

| category | rank_in_cat | app | reviews |
|---|---|---|---|
| COMMUNICATION | 1 | WhatsApp Messenger | 69119316 |
| COMMUNICATION | 2 | Messenger - Text and Video Chat for Free | 56646578 |
| COMMUNICATION | 3 | UC Browser - Fast Download Private & Secure | 17714850 |
| GAME | 1 | Clash of Clans | 44893888 |
| GAME | 2 | Subway Surfers | 27725352 |
| GAME | 3 | Clash Royale | 23136735 |
| SOCIAL | 1 | Facebook | 78158306 |
| SOCIAL | 2 | Instagram | 66577446 |
| SOCIAL | 3 | Snapchat | 17015352 |

**Why:** Without the dedup, an app listed 4 times would take several of the top 3 places in its category (Clash of Clans or Subway Surfers several times). Dedup first, rank second, filter last. The WHERE on category runs before the window, which is fine here because the ranking is per category anyway. Facebook leads SOCIAL, WhatsApp leads COMMUNICATION and Clash of Clans leads GAME.

**Complexity:** Two window sorts over about 10.8k rows.

**Common mistakes:** Ranking raw rows. Ranking the text column (as text, '9' > '10'). Using ROW_NUMBER for "ties share a rank".

**Learn more:** [sql-top-n](https://mahsa-agz.github.io/ds-handbook/#sql-top-n), [sql-dedup-cleaning](https://mahsa-agz.github.io/ds-handbook/#sql-dedup-cleaning)

</details>

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Next:** [Statistics, A/B testing and product](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_20/3_stats.ipynb)